# Window Decoding Tutorial

This notebook explains how **window decoding** works in deq using the
repetition code as a concrete example. Window decoding is a streaming
approach to quantum error correction where we decode overlapping
"windows" of gadgets rather than waiting for the entire circuit.

**Prerequisites:**
```bash
pip install deq[render] deq_runtime
playwright install chromium
```

---
## Part 1: Gadget Vocabulary

Before diving into window decoding, let us visualize the basic building
blocks — the gadget types in a distance-3 repetition code.

In [ ]:
import shutil
from pathlib import Path
import deq.proto.deq_jit_pb2 as jit_pb
from deq import cli

from window_decoding_tutorial import (
    show, render_gadget_types,
    cleanup_server, start_server, run_circuit, run_circuit_streaming,
    build_gtype_info, plot_timeline,
    generate_random_program, build_layout_from_program,
    plot_2d_snapshot, add_role_legend,
    precompute_trace_frames, render_trace_svg_frames,
    WindowTraceVisualizer, SVGPlayer,
)
import matplotlib.pyplot as plt

### Generate the JIT Library

We use the repetition code `.deq` files to compile a JIT library.


In [ ]:
jit_folder = Path("./window_tutorial_data")
if jit_folder.exists():
    shutil.rmtree(jit_folder)
jit_folder.mkdir(parents=True, exist_ok=True)

# Use the existing repetition code d=3 .deq file
rep_code_deq = str(Path("../../tests/circuit/repetition_code/repetition_code_d3.deq").resolve())

# Transpile to JIT library
jit_output = str(jit_folder / "library.deq.jit")
cli.jit.transpile(rep_code_deq, out=jit_output)

# Load the JIT library
with open(jit_output, "rb") as f:
    jit_library = jit_pb.JitLibrary.FromString(f.read())

print(f"Gadget types: {len(jit_library.gadget_types)}")
for gt in jit_library.gadget_types:
    print(f"  {gt.base.name}: {len(gt.base.measurements)} measurements, "
          f"{len(gt.base.inputs)} in, {len(gt.base.outputs)} out")


### Render Each Gadget Type

We render each gadget type in **realization view** to show the underlying
quantum circuit. Data qubits are arranged horizontally, and time flows
vertically (top to bottom).

In [ ]:
render_gadget_types(jit_library)

---
## Part 2: Basic Window Decoding

Window decoding works by dividing a circuit into overlapping **windows**.
Each window is decoded independently, and the results are stitched together.

Key concepts:
- **Commit region**: The gadgets whose corrections are finalized by a window
- **Buffer region**: Surrounding gadgets included in the window for decoder
  context but not committed
- **`buffer_radius`**: Minimum distance from the window boundary required for
  a gadget to be committed (controls buffer thickness)
- **`lookahead_radius`**: How far from the center additional gadgets may be
  committed in the same wave

Let us run a simple chain: `PrepareZ → Idle → Idle → MeasureZ` with the
window coordinator (`buffer_radius=1, lookahead_radius=0`) and visualize
the decoding progression.


In [ ]:
gtype_to_info = build_gtype_info(jit_library)

In [ ]:
trace_path = jit_folder / "trace_basic.pb"
port = start_server(jit_output, buffer_radius=1, lookahead_radius=0, trace_path=trace_path)

program = "PrepareZ 0\nIdle 0\nIdle 0\nMeasureZ 0"
print(f"Running:\n{program}")
run_circuit(jit_library, program, port, gtype_to_info)


### Visualize the Decoding Progression

We load the trace file and render snapshots at key timestamps.
Each snapshot shows gadget states with colors:
- **White**: Not yet executed
- **Gray**: Executed (outcomes loaded, waiting for decode)
- **Purple**: Committing (active decode window, being committed)
- **Gold**: Buffer (part of active window, providing context)
- **Green**: Committed (decode complete, corrections applied)


In [ ]:
viz = WindowTraceVisualizer(trace_path, jit_output)
timeline = viz.get_timeline_data(shot_index=0)

print(f"Gadgets: {len(timeline['gadgets'])}")
print(f"Windows: {len(timeline['windows'])}")
for w in timeline["windows"]:
    buffer = sorted(set(w['window']) - set(w['committing_gids']))
    print(f"  Leader gid={w['leader_gid']}: commits {w['committing_gids']}, buffer {buffer}")

layout = build_layout_from_program(program)
frames = precompute_trace_frames(viz)
svg_frames = render_trace_svg_frames(frames, layout)
print(f"\n{len(svg_frames)} frames:")

player = SVGPlayer(svg_frames)
display(player)

In [ ]:
# Render all frames as inline SVGs (viewable in git)
from IPython.display import display, SVG
for i, frame in enumerate(svg_frames):
    print(f"Frame {i}/{len(svg_frames)-1}: {frame.label}")
    display(SVG(data=frame.svg))

---
## Part 3: Free-Hop (Transversal) Gadgets

Transversal gadgets (like CNOT) have **zero syndrome extraction rounds**,
so they contribute 0 to the BFS distance in window construction.
They are absorbed into the commit region of whatever window contains them.

Let us run a 2-qubit circuit with a transversal CNOT.


In [ ]:
trace_path_3 = jit_folder / "trace_freehop.pb"
cleanup_server()
port = start_server(jit_output, buffer_radius=1, lookahead_radius=0, trace_path=trace_path_3)

program = (
    "PrepareZ 0\n"
    "PrepareZ 1\n"
    "TransversalCNOT 0 1\n"
    "Idle 0\n"
    "Idle 1\n"
    "MeasureZ 0\n"
    "MeasureZ 1"
)
print(f"Running:\n{program}")
run_circuit(jit_library, program, port, gtype_to_info)


In [ ]:
viz3 = WindowTraceVisualizer(trace_path_3, jit_output)
timeline3 = viz3.get_timeline_data(shot_index=0)

print(f"Windows: {len(timeline3['windows'])}")
for w in timeline3["windows"]:
    buffer = sorted(set(w['window']) - set(w['committing_gids']))
    print(f"  Leader gid={w['leader_gid']}: commits {w['committing_gids']}, buffer {buffer}")

layout3 = build_layout_from_program(program)
frames3 = precompute_trace_frames(viz3)
svg_frames3 = render_trace_svg_frames(frames3, layout3)
print(f"\n{len(svg_frames3)} frames:")
player3 = SVGPlayer(svg_frames3)
display(player3)

In [ ]:
# Render all frames as inline SVGs (viewable in git)
from IPython.display import display, SVG
for i, frame in enumerate(svg_frames3):
    print(f"Frame {i}/{len(svg_frames3)-1}: {frame.label}")
    display(SVG(data=frame.svg))

---
## Part 4: Distance-Based Commit Regions

The window coordinator uses **two radius parameters** to control how
windows are built and which gadgets get committed:

| Parameter | Default | Meaning |
|---|---|---|
| `buffer_radius` | 1 | Minimum hop-distance from the window boundary for a gadget to be committed. Larger values mean more decoder context (higher quality) but fewer gadgets committed per window. |
| `lookahead_radius` | = `buffer_radius` | How far from the center gadget other gadgets may also be committed in the same wave. `0` means only the center gadget is committed. |

The **effective window radius** (how far the BFS expands) is
`buffer_radius + lookahead_radius`.

**How it works:**
1. A decode request arrives for a *center* gadget (the leader).
2. The coordinator expands a BFS window of radius `buffer_radius + lookahead_radius`.
3. Within that window, it computes each gadget's distance to the nearest
   open boundary (edges not yet committed).
4. Gadgets within `lookahead_radius` of the center **and** at least
   `buffer_radius` from a boundary are committed.
5. The remaining window gadgets serve as buffer — they provide context
   but are not finalized.


### Effect of `buffer_radius`

We fix `lookahead_radius=0` (only the center is committed) and sweep
`buffer_radius` from 1 to 3. Larger `buffer_radius` means each window
carries more buffer gadgets, giving the decoder more context.


In [ ]:
long_program = (
    "PrepareZ 0\n"
    "Idle 0\n"
    "Idle 0\n"
    "Idle 0\n"
    "Idle 0\n"
    "Idle 0\n"
    "Idle 0\n"
    "MeasureZ 0"
)
long_layout = build_layout_from_program(long_program)

for cr in [1, 2, 3]:
    trace_path_cr = jit_folder / f"trace_cr{cr}.pb"
    cleanup_server()
    port = start_server(jit_output, buffer_radius=cr, lookahead_radius=0,
                        trace_path=trace_path_cr)
    print(f"\n=== buffer_radius={cr}, lookahead_radius=0 ===")
    run_circuit(jit_library, long_program, port, gtype_to_info)

    viz_cr = WindowTraceVisualizer(trace_path_cr, jit_output)
    tl_cr = viz_cr.get_timeline_data(shot_index=0)
    print(f"  Windows: {len(tl_cr['windows'])}")
    for w in tl_cr["windows"]:
        buffer = sorted(set(w["window"]) - set(w["committing_gids"]))
        print(f"    Leader gid={w['leader_gid']}: commits {w['committing_gids']}, buffer {buffer}")

    plot_timeline(tl_cr, title=f"buffer_radius={cr}, lookahead_radius=0")


In [ ]:
# Visualize the first decode window for each buffer_radius setting
from IPython.display import display, SVG

fig, axes = plt.subplots(1, 3, figsize=(18, 3))
for idx, cr in enumerate([1, 2, 3]):
    trace_path_cr = jit_folder / f"trace_cr{cr}.pb"
    viz_cr = WindowTraceVisualizer(trace_path_cr, jit_output)
    frames_cr = precompute_trace_frames(viz_cr)
    # Show the frame where the first window is actively decoding
    # (frame 1 is typically the first decode snapshot)
    frame_idx = min(1, len(frames_cr) - 1)
    plot_2d_snapshot(long_layout, frames_cr[frame_idx]["roles"],
                     title=f"buffer_radius={cr} — {frames_cr[frame_idx]['label']}",
                     ax=axes[idx])
add_role_legend(fig)
plt.tight_layout()
plt.subplots_adjust(bottom=0.22)
plt.show()


### Effect of `lookahead_radius`

The `lookahead_radius` controls how many neighboring gadgets around the
center can be committed in the **same wave**, provided they also satisfy
the `buffer_radius` boundary-distance requirement.

- `lookahead_radius=0`: Only the center gadget is committed per window →
  many small windows, maximum overlap.
- `lookahead_radius=1`: The center and its immediate neighbors may be
  committed → fewer, larger windows.
- `lookahead_radius=2`: Even more gadgets batched per window.

We fix `buffer_radius=1` and sweep `lookahead_radius`.


In [ ]:
for cenr in [0, 1, 2]:
    trace_path_cenr = jit_folder / f"trace_cenr{cenr}.pb"
    cleanup_server()
    port = start_server(jit_output, buffer_radius=1, lookahead_radius=cenr,
                        trace_path=trace_path_cenr)
    print(f"\n=== buffer_radius=1, lookahead_radius={cenr} ===")
    run_circuit(jit_library, long_program, port, gtype_to_info)

    viz_cenr = WindowTraceVisualizer(trace_path_cenr, jit_output)
    tl_cenr = viz_cenr.get_timeline_data(shot_index=0)
    print(f"  Windows: {len(tl_cenr['windows'])}")
    for w in tl_cenr["windows"]:
        buffer = sorted(set(w["window"]) - set(w["committing_gids"]))
        print(f"    Leader gid={w['leader_gid']}: commits {w['committing_gids']}, buffer {buffer}")

    plot_timeline(tl_cenr, title=f"buffer_radius=1, lookahead_radius={cenr}")


In [ ]:
# Compare first decode window across lookahead_radius values
fig, axes = plt.subplots(1, 3, figsize=(18, 3))
for idx, cenr in enumerate([0, 1, 2]):
    trace_path_cenr = jit_folder / f"trace_cenr{cenr}.pb"
    viz_cenr = WindowTraceVisualizer(trace_path_cenr, jit_output)
    frames_cenr = precompute_trace_frames(viz_cenr)
    frame_idx = min(1, len(frames_cenr) - 1)
    plot_2d_snapshot(long_layout, frames_cenr[frame_idx]["roles"],
                     title=f"lookahead_radius={cenr} — {frames_cenr[frame_idx]['label']}",
                     ax=axes[idx])
add_role_legend(fig)
plt.tight_layout()
plt.subplots_adjust(bottom=0.22)
plt.show()


### Combined Parameter Interaction

Let us visualize how both parameters interact with `buffer_radius=2`
and `lookahead_radius=2`. The effective window radius is 4 hops,
producing wide windows with large commit regions.


In [ ]:
trace_path_combined = jit_folder / "trace_combined.pb"
cleanup_server()
port = start_server(jit_output, buffer_radius=2, lookahead_radius=2,
                    trace_path=trace_path_combined)
print("=== buffer_radius=2, lookahead_radius=2 (effective window radius = 4) ===")
run_circuit(jit_library, long_program, port, gtype_to_info)

viz_combined = WindowTraceVisualizer(trace_path_combined, jit_output)
tl_combined = viz_combined.get_timeline_data(shot_index=0)
print(f"Windows: {len(tl_combined['windows'])}")
for w in tl_combined["windows"]:
    buffer = sorted(set(w["window"]) - set(w["committing_gids"]))
    print(f"  Leader gid={w['leader_gid']}: commits {w['committing_gids']}, buffer {buffer}")

plot_timeline(tl_combined, title="buffer_radius=2, lookahead_radius=2")


In [ ]:
frames_combined = precompute_trace_frames(viz_combined)
svg_frames_combined = render_trace_svg_frames(frames_combined, long_layout)
print(f"{len(svg_frames_combined)} frames")
player_combined = SVGPlayer(svg_frames_combined)
display(player_combined)


In [ ]:
# Render all combined frames as inline SVGs (viewable in git)
from IPython.display import display, SVG
for i, frame in enumerate(svg_frames_combined):
    print(f"Frame {i}/{len(svg_frames_combined)-1}: {frame.label}")
    display(SVG(data=frame.svg))


### Summary

| Configuration | Window radius | Commits per window | Tradeoff |
|---|---|---|---|
| `buffer_radius=1, lookahead_radius=0` | 1 | 1 gadget | Many small windows, maximum parallelism |
| `buffer_radius=2, lookahead_radius=0` | 2 | 1 gadget | More decoder context per window, same commit rate |
| `buffer_radius=1, lookahead_radius=1` | 2 | ~2 gadgets | Larger commit regions, fewer windows needed |
| `buffer_radius=2, lookahead_radius=2` | 4 | ~3–4 gadgets | Wide windows, large commit regions, high quality |

**Rule of thumb:** Increase `buffer_radius` for better decoding quality
(more buffer context). Increase `lookahead_radius` to commit more gadgets
per window (fewer total windows, lower overhead, but also lower parallelism).


---
## Part 5: Batch vs Stream Decoding

The window coordinator supports two execution modes:

- **Batch mode**: All gadgets are executed first, then all decode calls
  arrive simultaneously. The coordinator can decode non-overlapping
  windows in parallel.
- **Stream mode**: Gadgets are executed and decoded incrementally as
  measurements arrive. The coordinator starts decoding as soon as
  enough gadgets form a window, overlapping decoding with execution.

Let us compare both modes on a simple 5-idle chain.

In [ ]:
n_idles = 5
chain_parts = ["PrepareZ 0"]
for i in range(n_idles):
    chain_parts.append("Idle 0")
chain_parts.append("MeasureZ 0")
chain_program = "\n".join(chain_parts)
print(f"Circuit:\n{chain_program}")
print(f"{n_idles} idle gadgets + prepare + measure = {n_idles + 2} total")


### Batch Decoding

Execute all gadgets first, then submit all decode calls at once.

In [ ]:
trace_batch = jit_folder / "trace_batch.pb"
cleanup_server()
port = start_server(jit_output, buffer_radius=1, lookahead_radius=1, trace_path=trace_batch)
print("Running batch decode...")
run_circuit(jit_library, chain_program, port, gtype_to_info)

In [ ]:
viz_batch = WindowTraceVisualizer(trace_batch, jit_output)
tl_batch = viz_batch.get_timeline_data(shot_index=0)
print(f"Windows: {len(tl_batch['windows'])}")
for w in tl_batch["windows"]:
    buffer = sorted(set(w['window']) - set(w['committing_gids']))
    print(f"  Leader gid={w['leader_gid']}: commits {w['committing_gids']}, buffer {buffer}")

plot_timeline(tl_batch, "Batch Decode (all gadgets executed first)")

layout_batch = build_layout_from_program(chain_program)
frames_batch = precompute_trace_frames(viz_batch)
svg_frames_batch = render_trace_svg_frames(frames_batch, layout_batch)
print(f"\n{len(svg_frames_batch)} trace frames")

player_batch = SVGPlayer(svg_frames_batch)
display(player_batch)

In [ ]:
# Render all batch frames as inline SVGs (viewable in git)
from IPython.display import display, SVG
for i, frame in enumerate(svg_frames_batch):
    print(f"Frame {i}/{len(svg_frames_batch)-1}: {frame.label}")
    display(SVG(data=frame.svg))

### Stream Decoding

Execute and decode gadgets one at a time with a 150ms delay between
them. Since the mock decoder takes ~100ms, the coordinator can start
decoding early windows while later gadgets are still being executed.

In [ ]:
trace_stream = jit_folder / "trace_stream.pb"
cleanup_server()
port = start_server(jit_output, buffer_radius=1, lookahead_radius=1, trace_path=trace_stream)
print("Running stream decode (150ms between gadgets)...")
run_circuit_streaming(jit_library, chain_program, port, gtype_to_info,
                      delay_seconds=0.15)

In [ ]:
viz_stream = WindowTraceVisualizer(trace_stream, jit_output)
tl_stream = viz_stream.get_timeline_data(shot_index=0)
print(f"Windows: {len(tl_stream['windows'])}")
for w in tl_stream["windows"]:
    buffer = sorted(set(w['window']) - set(w['committing_gids']))
    print(f"  Leader gid={w['leader_gid']}: commits {w['committing_gids']}, buffer {buffer}")

plot_timeline(tl_stream, "Stream Decode (150ms between gadgets)")

layout_stream = build_layout_from_program(chain_program)
frames_stream = precompute_trace_frames(viz_stream)
svg_frames_stream = render_trace_svg_frames(frames_stream, layout_stream)
print(f"\n{len(svg_frames_stream)} trace frames")

player_stream = SVGPlayer(svg_frames_stream)
display(player_stream)

In [ ]:
# Render all stream frames as inline SVGs (viewable in git)
from IPython.display import display, SVG
for i, frame in enumerate(svg_frames_stream):
    print(f"Frame {i}/{len(svg_frames_stream)-1}: {frame.label}")
    display(SVG(data=frame.svg))

### Comparison

In **batch mode**, all decode windows start at roughly the same time
(after all gadgets are executed) and run in parallel waves.

In **stream mode**, decode windows start as soon as their gadgets arrive
and complete incrementally. The total wall-clock time can be lower because
decoding overlaps with execution — the decoder processes early gadgets
while later ones are still being measured.

---
## Part 6: Random Circuit Experiment

Finally, let us try a larger random circuit with multiple qubits and
transversal CNOT gates. This shows how the window coordinator handles
realistic circuit topologies.

In [ ]:
random_program, random_layout = generate_random_program(n_qubits=8, n_gates=64, seed=42)
n_cnots = sum(1 for g in random_layout["gadgets"].values() if g["kind"] == "cnot")
print(f"Random circuit: {random_layout['n_qubits']} qubits, "
      f"{len(random_layout['gadgets'])} gadgets ({n_cnots} CNOTs), "
      f"{random_layout['max_cycle']+1} clock cycles")
for part in random_program.split("; "):
    print(f"  {part}")

In [ ]:
trace_random = jit_folder / "trace_random.pb"
cleanup_server()
port = start_server(jit_output, buffer_radius=1, lookahead_radius=3, trace_path=trace_random)
print("Running random circuit...")
run_circuit(jit_library, random_program, port, gtype_to_info)

viz_random = WindowTraceVisualizer(trace_random, jit_output)
tl_random = viz_random.get_timeline_data(shot_index=0)
print(f"\nGadgets: {len(tl_random['gadgets'])}")
print(f"Windows: {len(tl_random['windows'])}")

plot_timeline(tl_random, "Random Circuit (8 qubits, 64 gates, buffer_radius=1, lookahead_radius=3)")


In [ ]:
# Static circuit layout (all committed)
all_committed = {gid: "committed" for gid in random_layout["gadgets"]}
fig, ax = plt.subplots(figsize=(max(8, (random_layout["max_cycle"] + 1) * 1.0),
                                max(3, random_layout["n_qubits"] * 0.9)))
plot_2d_snapshot(random_layout, all_committed, title="Circuit Layout", ax=ax)
add_role_legend(fig)
plt.tight_layout()
fig.subplots_adjust(bottom=0.12)
plt.show()

# Interactive trace player
frames = precompute_trace_frames(viz_random)
svg_frames = render_trace_svg_frames(frames, random_layout)
print(f"Precomputed {len(svg_frames)} SVG frames for interactive player")
player = SVGPlayer(svg_frames)
display(player)

In [ ]:
# Render all frames as inline SVGs (viewable in git)
from IPython.display import display, SVG
for i, frame in enumerate(svg_frames):
    print(f"Frame {i}/{len(svg_frames)-1}: {frame.label}")
    display(SVG(data=frame.svg))

---
## Cleanup

In [ ]:
cleanup_server()
print("Server stopped. Tutorial complete!")